# Báo cáo Project
Lớp TTNT-172886, Nhóm G14

## 1. Thông tin chung

### Thành viên
- Võ Hữu Trí Dũng 20235057
- Nguyễn Văn Tiến Dũng 20235306
- Nguyễn Đức Huân 20235338

### Lịch thực hiện
- W2: Đăng ký nhóm 
- W3-4: Đề xuất project (26/9)
- W08: Báo cáo tiến độ giữa kỳ (24/10)
- W15: Hoàn thành và gửi báo cáo project (12/12)
- W16-18: Trình bày project, Q&A

## 2. Đề xuất project (W3)

### Bài toán
Dự án nghiên cứu bài toán **học chính sách chọn nước đi trong cờ vua** bằng phương pháp học có giám sát và chưng cất tri thức. Với đầu vào là một trạng thái bàn cờ tiêu chuẩn chứa đầy đủ thông tin để xác định các nước hợp lệ, mô hình cần sinh ra phân phối xác suất trên tập nước đi hợp lệ và lựa chọn nước có xác suất cao nhất. Khác với các engine truyền thống, mô hình dự đoán nước đi trực tiếp bằng một lần suy luận, không mở rộng cây tìm kiếm trong quá trình sử dụng.

Dữ liệu giám sát gồm hai nguồn bổ sung cho nhau: nước đi thực tế trong các ván đấu chất lượng cao và phân phối xác suất mềm được xây dựng từ đánh giá của Stockfish. Nước đi của người chơi phản ánh quyết định đã xảy ra, còn phân phối Stockfish cung cấp thêm thông tin về nhiều phương án tốt và mức độ ưu tiên tương đối giữa chúng.

Câu hỏi thực nghiệm chính của dự án là: **với cùng kiến trúc, dữ liệu đầu vào và ngân sách huấn luyện, việc kết hợp nhãn nước đi của người chơi với phân phối mềm từ Stockfish có cải thiện chất lượng chọn nước so với mô hình chỉ học từ nước đi của người chơi hay không?** Kết quả được phân tích riêng ở khai cuộc, trung cuộc và tàn cuộc để xác định mức cải thiện có đồng đều giữa các giai đoạn hay không. Phạm vi dự án tập trung vào policy learning và knowledge distillation.

### Phương pháp dự kiến
Quy trình thực hiện gồm bốn khối chính: xây dựng dữ liệu, biểu diễn trạng thái và hành động, huấn luyện mạng chính sách, và đánh giá mô hình.

1. **Xây dựng và gắn nhãn dữ liệu**
   - Sử dụng các ván cờ tiêu chuẩn chất lượng cao từ **Lichess Elite Database**, bắt đầu từ dữ liệu tháng 12/2021 và tuân theo tiêu chí tuyển chọn của nguồn.
   - Loại bỏ các bản ghi không hợp lệ, không thuộc biến thể standard, thiếu chuỗi nước đi hoặc không thể phân tích. Dữ liệu được chia theo **ván cờ**, không chia theo từng vị trí, với tỷ lệ 90% train, 5% validation và 5% test nhằm tránh rò rỉ các trạng thái liên tiếp của cùng một ván giữa các tập.
   - Dự kiến thu thập 1.500.000 vị trí, cân bằng thành 500.000 vị trí khai cuộc, 500.000 vị trí trung cuộc và 500.000 vị trí tàn cuộc. Mỗi ván lấy tối đa một vị trí ở mỗi giai đoạn để hạn chế sự phụ thuộc giữa các mẫu.
   - Mỗi mẫu lưu trạng thái bàn cờ trước nước đi, nước thực tế của người chơi và thông tin nguồn. Nước người chơi được dùng làm **nhãn cứng**; Stockfish chạy ngoại tuyến với `MultiPV = 8` để đánh giá tối đa tám phương án tốt nhất và tạo **phân phối nhãn mềm**. Các điểm centipawn được chuyển thành xác suất bằng softmax với nhiệt độ dự kiến 100 centipawn.

2. **Biểu diễn trạng thái và không gian hành động**
   - Bàn cờ được chuẩn hóa theo góc nhìn của bên tới lượt và mã hóa thành tensor kích thước $17 \times 8 \times 8$. Trong đó, 12 mặt phẳng biểu diễn các loại quân của hai bên, 4 mặt phẳng biểu diễn quyền nhập thành và 1 mặt phẳng biểu diễn ô bắt tốt qua đường.
   - Nước đi được ánh xạ vào không gian hành động cố định kiểu AlphaZero có kích thước $8 \times 8 \times 73 = 4{,}672$, bao gồm các nước đi theo hướng, nước mã và các trường hợp phong cấp.
   - Tại mỗi trạng thái, mặt nạ nước hợp lệ được áp dụng trước softmax để loại bỏ hoàn toàn xác suất của các hành động không hợp lệ. Cùng một quy tắc mã hóa và mặt nạ được sử dụng trong huấn luyện, đánh giá và suy luận.

3. **Kiến trúc và huấn luyện mô hình**
   - Mạng chính sách sử dụng backbone **ResNet-16×256**, gồm 16 residual blocks với 256 kênh đặc trưng. Policy head dùng phép tích chập $1 \times 1$ để sinh 73 kênh và làm phẳng thành 4.672 logits tương ứng với toàn bộ không gian hành động.
   - Mô hình baseline sử dụng cùng kiến trúc và chỉ tối ưu cross-entropy theo nước đi thực tế của người chơi. Mô hình chính kết hợp mất mát nhãn cứng và distillation loss từ phân phối Stockfish: $\mathcal{L}=0.5\mathcal{L}_{human}+0.5\mathcal{L}_{KD}$.
   - Cấu hình dự kiến gồm AdamW, learning rate $3 \times 10^{-4}$, weight decay $10^{-4}$, cosine decay, batch hiệu dụng 512 và tối đa 8 epoch. Mô hình tốt nhất được lựa chọn trên validation set; mixed precision và gradient accumulation được sử dụng nếu cần để phù hợp với tài nguyên GPU.

4. **Đánh giá và so sánh**
   - So sánh công bằng mô hình **human-only baseline** với mô hình **hybrid human + Stockfish** trên cùng test set, cùng cách mã hóa, mặt nạ nước hợp lệ và kiến trúc.
   - Đánh giá độ khớp chính sách bằng Top-1, Top-3 và KL divergence; đánh giá chất lượng thực tế của nước đi bằng centipawn loss khi phân tích lại với Stockfish ở ngân sách cao hơn.
   - Báo cáo riêng kết quả cho khai cuộc, trung cuộc và tàn cuộc. Nếu đủ tài nguyên, tổ chức các ván đấu có kiểm soát giữa các mô hình, đổi màu theo cặp và thống kê số ván thắng/hòa/thua.

### Phân công
- **Võ Hữu Trí Dũng — Phụ trách dữ liệu:**
  - Thu thập và kiểm tra các ván cờ từ Lichess Elite Database; xây dựng bộ lọc để loại bỏ dữ liệu không hợp lệ hoặc không đáp ứng tiêu chí của dự án.
  - Phân chia dữ liệu theo ván thành train/validation/test, khử trùng lặp giữa các tập và lấy mẫu cân bằng theo ba giai đoạn khai cuộc, trung cuộc, tàn cuộc.
  - Xây dựng quy trình chuẩn hóa trạng thái bàn cờ, mã hóa nước đi và kiểm thử các trường hợp đặc biệt như nhập thành, bắt tốt qua đường và phong cấp.
  - Chạy Stockfish để tạo nhãn mềm, quản lý các shard dữ liệu và lập manifest ghi lại nguồn, cấu hình xử lý, phiên bản công cụ và hash của từng tập dữ liệu.

- **Nguyễn Văn Tiến Dũng — Phụ trách huấn luyện:**
  - Cài đặt mạng chính sách ResNet-16×256, policy head, legal move masking và các hàm mất mát cho nhãn cứng và knowledge distillation.
  - Xây dựng hai cấu hình thí nghiệm chính: mô hình human-only baseline và mô hình hybrid kết hợp nhãn người chơi với phân phối Stockfish.
  - Thiết lập quy trình huấn luyện, lựa chọn optimizer và learning-rate scheduler; thực hiện benchmark để xác định microbatch, gradient accumulation và các tham số phù hợp với tài nguyên GPU.
  - Theo dõi loss và các chỉ số trên validation set, lưu checkpoint và log thí nghiệm, bảo đảm quá trình huấn luyện có thể tiếp tục và tái lập.

- **Nguyễn Đức Huân — Phụ trách đánh giá:**
  - Xây dựng pipeline suy luận và đánh giá thống nhất cho các mô hình trên test set, luôn áp dụng cùng cách mã hóa và mặt nạ nước hợp lệ.
  - Tính toán các chỉ số Top-1, Top-3, KL divergence và centipawn loss; phân tích kết quả riêng cho khai cuộc, trung cuộc và tàn cuộc.
  - Tổ chức đánh giá theo ván đấu nếu đủ tài nguyên, sử dụng cùng điều kiện thời gian hoặc giới hạn suy luận, đổi màu theo cặp và thống kê thắng/hòa/thua.
  - So sánh các mô hình, tổng hợp bảng kết quả, trực quan hóa số liệu và phân tích các trường hợp dự đoán sai hoặc nước đi có chất lượng thấp.

## 3. Tiến độ giữa kỳ (W8)

### Chương trình


### Phân tích kết quả, vấn đề gặp phải


## 4. Cập nhật kết quả cuối kỳ (W15)

### Chi tiết phương pháp, dữ liệu 


### Chương trình


### Phân tích, đánh giá kết quả


### Cập nhật phân công, khối lượng công việc
<!-- công việc của các thành viên, tỷ lệ đóng góp của các thành viên -->
